In [ ]:
!pip install -q numpyro@git+https://github.com/pyro-ppl/numpyro

: 

# Composing Gaussian Processes with NumPyro using GPJax

[GPJax](https://docs.jaxgaussianprocesses.com/) is a Gaussian process (GP) library built on JAX and Equinox. In this notebook, we use GPJax's GP components inside a NumPyro model and run MCMC over everything jointly — regression coefficients and GP hyperparameters alike.

The model is a semiparametric spatial regression:

$$y(\mathbf{x}) = \underbrace{\mathbf{w}^\top \mathbf{x} + b}_{\text{linear trend}} + \underbrace{f(\mathbf{x})}_{\text{GP residual}} + \epsilon$$

We'll use NumPyro to handle the linear part, GPJax for the GP components, and NUTS to draw samples from the joint posterior.

In [ ]:
import gpjax as gpx
import matplotlib as mpl
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.random as jr

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive

jax.config.update("jax_enable_x64", True)
assert numpyro.__version__.startswith("0.20.1")

## Data simulation

We generate 200 points on a $[0, 5] \times [0, 5]$ grid. The true signal has two parts: a linear trend $y_{\text{lin}} = 2x_1 - x_2 + 1.5$ and a nonlinear residual $y_{\text{res}} = \sin(x_1)\cos(x_2)$, plus Gaussian noise $\epsilon \sim \mathcal{N}(0, 0.1^2)$.

In [ ]:
N = 200
key = jr.key(123)
keys = jr.split(key, 8)

X = jr.uniform(keys[0], shape=(N, 2), minval=0.0, maxval=5.0)

# True generative components
true_slope = jnp.array([2.0, -1.0])
true_intercept = 1.5
y_lin = X @ true_slope + true_intercept
y_res = jnp.sin(X[:, 0]) * jnp.cos(X[:, 1])

latent_signal = y_lin + y_res
noise_stddev = 0.1
y = latent_signal + noise_stddev * jr.normal(keys[1], shape=latent_signal.shape)

## Building the GP inside the NumPyro model

GPJax parameters may be initialised with JAX arrays and so any value drawn from `numpyro.sample` can be passed in as a kernel or likelihood hyperparameter. This is in contrast with GPJax versions <0.14 where in a registration step was required. Since v0.14, the integration is much simpler.

In the model below we draw the lengthscale, variance, and observation standard deviation as ordinary NumPyro sample sites and then construct the GPJax `Prior`, `Likelihood`, and `Posterior` from those draws on each model evaluation. The GP's marginal log-likelihood on the residuals is added to the joint density via `numpyro.factor`. For predictions, `gp_posterior.predict(X_new, train_data=D_resid)` gives the GP's predictive distribution at new locations and the total prediction is the linear trend plus the GP residual.

In [ ]:
def joint_model(X, Y, X_new=None):
    # Linear trend parameters
    slope = numpyro.sample("slope", dist.Normal(0.0, 5.0).expand([2]))
    intercept = numpyro.sample("intercept", dist.Normal(0.0, 5.0))

    # GP hyperparameters
    lengthscale = numpyro.sample("lengthscale", dist.LogNormal(0.0, 1.0))
    variance = numpyro.sample("variance", dist.LogNormal(0.0, 1.0))
    obs_stddev = numpyro.sample("obs_stddev", dist.LogNormal(0.0, 1.0))

    # GP construction with raw JAX scalars from numpyro.sample
    kernel = gpx.kernels.Matern32(
        active_dims=[0, 1], lengthscale=lengthscale, variance=variance
    )
    meanf = gpx.mean_functions.Constant()
    gp_prior = gpx.gps.Prior(mean_function=meanf, kernel=kernel)
    likelihood = gpx.likelihoods.Gaussian(
        num_datapoints=X.shape[0], obs_stddev=obs_stddev
    )
    gp_posterior = gp_prior * likelihood

    trend = X @ slope + intercept

    if Y is not None:
        residuals = (Y - trend).reshape(-1, 1)
        D_resid = gpx.Dataset(X=X, y=residuals)
        mll = gpx.objectives.conjugate_mll(gp_posterior, D_resid)
        numpyro.factor("gp_log_lik", mll)

    if X_new is not None and Y is not None:
        residuals = (Y - trend).reshape(-1, 1)
        D_resid = gpx.Dataset(X=X, y=residuals)

        latent_dist = gp_posterior.predict(X_new, train_data=D_resid)
        f_new = numpyro.sample("f_new", latent_dist).reshape((-1, 1))

        total_prediction = (X_new @ slope + intercept).reshape(-1, 1) + f_new
        numpyro.deterministic("y_pred", total_prediction)

## MCMC inference

NUTS samples from the joint posterior over the linear parameters and the GP hyperparameters.

In [ ]:
nuts_kernel = NUTS(joint_model)
mcmc = MCMC(nuts_kernel, num_warmup=1500, num_samples=2000, num_chains=1)
mcmc.run(keys[2], X, y)
mcmc.print_summary()

### Inspecting the samples

Let's take a look at the posterior samples for the linear coefficients and the GP hyperparameters. We can use `arviz` to visualize the trace and summary statistics.

In [ ]:
samples = mcmc.get_samples()

param_info = [
    ("slope[0]", samples["slope"][:, 0], true_slope[0]),
    ("slope[1]", samples["slope"][:, 1], true_slope[1]),
    ("intercept", samples["intercept"], true_intercept),
    ("obs_stddev", samples["obs_stddev"], noise_stddev),
]

n_params = len(param_info)
fig, axes = plt.subplots(n_params, 2, figsize=(12, 2 * n_params))

for i, (name, chain, true_val) in enumerate(param_info):
    axes[i, 0].hist(
        chain, bins=40, density=True, alpha=0.7, color="C0", edgecolor="none"
    )
    if true_val is not None:
        axes[i, 0].axvline(
            true_val, color="C3", ls="--", lw=1.5, label=f"true = {true_val}"
        )
        axes[i, 0].legend(fontsize=8)
    axes[i, 0].set_ylabel(name, fontsize=10)
    axes[i, 0].set_yticks([])

    axes[i, 1].plot(chain, alpha=0.4, color="C0", lw=0.3)
    if true_val is not None:
        axes[i, 1].axhline(true_val, color="C3", ls="--", lw=1.5)

axes[0, 0].set_title("Posterior density")
axes[0, 1].set_title("Trace")
fig.suptitle("Posterior traces (dashed lines = true values)", fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

## Posterior predictions

`Predictive` generates predictions for each posterior sample, combining the linear trend with the GP residual.

In [ ]:
samples = mcmc.get_samples()
predictive = Predictive(joint_model, samples, return_sites=["y_pred"])
preds = predictive(keys[3], X=X, Y=y, X_new=X)["y_pred"]
mean_pred = jnp.mean(preds, axis=0)

rmse = jnp.sqrt(jnp.mean((mean_pred.flatten() - latent_signal.flatten()) ** 2))
print(f"Joint Model RMSE (vs true signal): {rmse:.4f}")

## Visualisation

True signal vs. the joint model’s posterior mean on a 2D grid.

In [ ]:
n_grid = 30
x1 = jnp.linspace(0, 5, n_grid)
x2 = jnp.linspace(0, 5, n_grid)
X1, X2 = jnp.meshgrid(x1, x2)
X_grid = jnp.column_stack([X1.ravel(), X2.ravel()])

y_grid_true = (X_grid @ true_slope + true_intercept) + (
    jnp.sin(X_grid[:, 0]) * jnp.cos(X_grid[:, 1])
)

preds_grid = predictive(keys[4], X=X, Y=y, X_new=X_grid)["y_pred"]
mean_pred_grid = jnp.mean(preds_grid, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

vmin = min(y_grid_true.min(), mean_pred_grid.min())
vmax = max(y_grid_true.max(), mean_pred_grid.max())
levels = jnp.linspace(vmin, vmax, 20)

axes[0].tricontourf(
    X_grid[:, 0], X_grid[:, 1], y_grid_true, levels=levels, cmap="magma"
)
axes[0].set_title("True Signal")

axes[1].tricontourf(
    X_grid[:, 0],
    X_grid[:, 1],
    mean_pred_grid.flatten(),
    levels=levels,
    cmap="magma",
)
axes[1].set_title(f"Joint Model (RMSE: {rmse:.2f})")

fig.colorbar(
    axes[0].collections[0],
    ax=axes.tolist(),
    format=mpl.ticker.FormatStrFormatter("%d"),
)

for ax in axes:
    ax.set_xlabel("$x_1$")
    ax.scatter(X[:, 0], X[:, 1], c="white", s=5, alpha=0.3, edgecolors="none")